In [35]:
from transformers import pipeline
import pandas as pd
import os

os.chdir("/Users/connorrust/Library/CloudStorage/Box-Box/Covid Policies/Data")
data = pd.read_csv("MN_Health_short.csv")

text = data["Text"].str.slice(0,50)
lst = text.to_list()

hypothesis_template = "This text is about {}"
classes_verbalized = ["covid", "health", "other"]

zeroshot_classifier = pipeline("zero-shot-classification", 
                               model="mlburnham/Political_DEBATE_DeBERTa_large_v1.1", 
                               device = "mps")  # change the model identifier here

output = zeroshot_classifier(lst, classes_verbalized, hypothesis_template=hypothesis_template, multi_label=True)

clean_output = []

for dct in output:
    nd = {}
    nd["sequence"] = dct["sequence"]
    for idx, label in enumerate(dct["labels"]):
        nd[label] = dct["scores"][idx]
    clean_output.append(nd)

df = pd.DataFrame(clean_output)

Device set to use mps


In [37]:
df

,sequence,health,covid,other
0,The Minnesota Department of Health today annou...,1.000000,0.999999,0.000009
1,The Minnesota Department of Health (MDH) today...,1.000000,0.999939,0.000009
2,This case marks the first documented instance ...,0.003184,0.000064,0.000019
3,A rapidly growing outbreak of variant COVID-19...,1.000000,1.000000,0.000022
4,As health officials continue to report more ca...,1.000000,1.000000,0.000016
5,Reflecting ongoing concerns about the potentia...,0.982775,0.000713,0.000067
6,May 12 will be the last day of testing at the ...,1.000000,1.000000,0.000009
7,Participating health plans starting this work ...,1.000000,0.000001,0.000027
8,The Minnesota Department of Health (MDH) today...,1.000000,0.999939,0.000009


In [38]:
from scipy.special import softmax
num_cols = df.columns.difference(['sequence'])
df2 = df.copy(deep=False)  
df2[num_cols] = softmax(df[num_cols].values, axis=1)
df2

,sequence,health,covid,other
0,The Minnesota Department of Health today annou...,0.422318,0.422318,0.155364
1,The Minnesota Department of Health (MDH) today...,0.422329,0.422303,0.155368
2,This case marks the first documented instance ...,0.334032,0.332991,0.332977
3,A rapidly growing outbreak of variant COVID-19...,0.422317,0.422317,0.155365
4,As health officials continue to report more ca...,0.422318,0.422318,0.155364
5,Reflecting ongoing concerns about the potentia...,0.571810,0.214164,0.214026
6,May 12 will be the last day of testing at the ...,0.422318,0.422318,0.155364
7,Participating health plans starting this work ...,0.576113,0.211941,0.211946
8,The Minnesota Department of Health (MDH) today...,0.422329,0.422303,0.155368


In [41]:
softmax([1, 1, 0.0000009])

array([0.42231874, 0.42231874, 0.15536252])

In [5]:

output2 = zeroshot_classifier(lst, classes_verbalized, 
                              hypothesis_template=hypothesis_template, 
                              multi_label=True)

clean_output = []

for dct in output2:
    nd = {}
    nd["sequence"] = dct["sequence"]
    for idx, label in enumerate(dct["labels"]):
        nd[label] = dct["scores"][idx]
    clean_output.append(nd)

df2 = pd.DataFrame(clean_output)


In [32]:
from scipy.special import softmax

df[["covid", "other"]].apply(lambda row: softmax(row), axis = 1)
# softmax(df2.to_numpy(, axis = 1)
# df2.apply(lambda row: softmax(row), axis = 1)


0     [0.7310567712812766, 0.2689432287187235]
1    [0.7310449227902115, 0.26895507720978845]
2    [0.5000112290304041, 0.49998877096959593]
3     [0.7310541765610605, 0.2689458234389394]
4     [0.7310554719397776, 0.2689445280602224]
5     [0.5001616503557739, 0.4998383496442262]
6     [0.7310567685503638, 0.2689432314496361]
7           [0.4999935409069, 0.5000064590931]
8    [0.7310449227902115, 0.26895507720978845]
dtype: object

In [33]:
df

,sequence,covid,other
0,The Minnesota Department of Health today annou...,0.999999,1.452811e-06
1,The Minnesota Department of Health (MDH) today...,0.999983,1.653233e-05
2,This case marks the first documented instance ...,0.646615,3.533848e-01
3,A rapidly growing outbreak of variant COVID-19...,0.999999,1.194613e-06
4,As health officials continue to report more ca...,0.999999,9.713034e-07
5,Reflecting ongoing concerns about the potentia...,0.770522,2.294779e-01
6,May 12 will be the last day of testing at the ...,0.999999,7.824493e-07
7,Participating health plans starting this work ...,0.191590,8.084098e-01
8,The Minnesota Department of Health (MDH) today...,0.999983,1.653233e-05
